# 02 — Warehouse access and training datasets

Notebook 01 built the **warehouse**. This notebook turns it into a **versioned training dataset** the rest of the pipeline (preprocessing, models, training, evaluation, inference) consumes.

## The architecture in one picture

A *dataset* is two things:

1. A **recipe** — `FeatureSelection` + date range + locations + QC filter. Pure Python config, lives in this repo, in git.
2. A **materialisation** — the parquet payload produced by applying the recipe to the warehouse at a specific point in time, plus a `manifest.json` capturing the warehouse state, the git SHA, and a SHA-256 of the parquet for tamper-detection.

The two-layer split is what lets us answer questions like:

* "What if we ingest more ground truth?" → same recipe, new materialisation, bumped version.
* "What if we add a new variable?" → new recipe, new artifact family.
* "What if we want fewer variables for an ablation?" → derived recipe, separate materialisation.
* "What if upstream data was wrong?" → fix in warehouse, re-run recipe, new materialisation.

## Where snapshots live

Each snapshot is a directory with `dataset.parquet` + `manifest.json`. Locally we write to `data/training_snapshots/<name>_<version>/`; from there we upload to **W&B Artifacts** of type `training_dataset`. Every training run consumes a specific artifact version, so the lineage graph in W&B answers "which model used which dataset" automatically.

In [ ]:
# ============================================================
# Colab bootstrap (no-op when run locally).
# ------------------------------------------------------------
# First-time setup on Colab:
#   1. Create a GitHub Personal Access Token (PAT) at
#      https://github.com/settings/tokens with `repo` scope.
#      The repository is private, so the clone needs this token
#      (or an SSH key Colab knows about, which is more fiddly).
#   2. Add the token under Tools → Secrets in Colab with name
#      `GITHUB_PAT` and toggle "Notebook access" on.
#   3. Sign in with a Google account that has BigQuery read access
#      to `solar-irradiation-estimation` when prompted.
# ============================================================
import os
import sys

if "google.colab" in sys.modules:
    REPO = "Marconi-Lab/Solar_irradiation"
    BRANCH = "jm/add_model"

    if not os.path.exists("/content/Solar_irradiation/.git"):
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_PAT")
            clone_url = f"https://{token}@github.com/{REPO}.git"
            print("Cloning with Colab secret 'GITHUB_PAT'.")
        except Exception:
            clone_url = f"git@github.com:{REPO}.git"
            print(
                "Colab secret 'GITHUB_PAT' not set — trying SSH. If the "
                "clone fails, follow the PAT setup steps above and re-run."
            )
        !git clone -q -b {BRANCH} {clone_url} /content/Solar_irradiation

    %cd /content/Solar_irradiation
    !pip install -q -e . 2>&1 | tail -3

    from google.colab import auth
    auth.authenticate_user()
    !gcloud config set project solar-irradiation-estimation 2>/dev/null
    print("Colab setup complete.")

### Inputs, outputs, and prerequisites

| | |
|---|---|
| **Inputs** | A populated warehouse (NB 01) and GCP application-default credentials. |
| **Outputs** | A snapshot directory on disk: `data/training_snapshots/<name>_<version>/{dataset.parquet, manifest.json}`. The same snapshot is optionally uploaded to W&B as an artifact of type `training_dataset`. |
| **Prereqs** | Env vars: none required to materialise locally. `WANDB_API_KEY` (in `.env`) needed only if `LOG_TO_WANDB=True`. |
| **Consumed by** | NB 03 (preprocessing) reads the snapshot directly via `load_snapshot(...)`. |

**Defaults are safe.** `LOG_TO_WANDB=False` and `DOWNLOAD_FROM_WANDB=False` keep the notebook offline-with-respect-to-W&B; you only need W&B if you want cross-machine artifact sharing.

## 0 — Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# Make the in-repo `src/` importable when running from `notebooks/`.
src_path = (Path.cwd() / "../../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load .env so WANDB_API_KEY (if present) is picked up automatically.
from susse.env import load_project_env
load_project_env()

print("Python", sys.version.split()[0])


In [ ]:
from susse.warehouse_ops.io import BigQueryClient, TableRefs

bq = BigQueryClient()
tables = TableRefs(config=bq.config)
print(f"Connected to {bq.config.project_id}.{bq.config.dataset}.")


## 1 — Define the recipe

`FeatureSelection` enumerates which variables to materialise per source, plus the QC filter and which satellite-irradiance series to include. Validation against the warehouse catalog runs at construction time, so a typo here surfaces as a `ValueError` *before* you spend a query on it.

For **v1** we pick a deliberately small recipe: ground GHI as the target, NASA + CAMS satellite GHI as the primary features, and three NASA atmospheric covariates. Future recipe versions will add MERRA-2 once it has full per-station coverage, and MODIS once the corresponding ingest is wired up.

### What goes into a `FeatureSelection`

Five fields, all type-checked against the warehouse:

- **`nasa_variable_ids` / `cams_variable_ids` / `merra_variable_ids` / `modis_variable_ids`** — auxiliary variables to materialise from each source's long-format table. Each is a tuple of `variable_id` strings; the discovery cell below lists what's available.
- **`include_satellite_irradiance`** — which sources of satellite GHI/DHI/DNI to pull from the wide `irradiance_daily` table. Today only `Source.NASA_POWER` and `Source.CAMS` write there; MERRA-2 and MODIS contribute features only.
- **`qc_levels`** — keep only ground rows whose `qc_level` tag matches one of these. NB 01's qc-levels query showed which tags exist in the warehouse; for v1 we only train on `pass`.

Aux columns come back **source-prefixed** in the output frame (e.g. `nasa_aod_550`, not `aod_550`) so MERRA-2's eventual `aod_550_analysis` won't collide with NASA POWER's `aod_550`. The y-target is always `y_ghi_kwh_m2_day`.

In [ ]:
# Browse what's available, grouped by source. Every variable_id you write
# into a FeatureSelection below must appear in this dataframe — typos
# raise a clear ValueError at construction.
from susse.warehouse_ops.population import VariableCatalog, variables_to_dataframe
from susse.warehouse_ops.population.types import Source

catalog_df = variables_to_dataframe(VariableCatalog.all_variables())
print("Sources:", [s.value for s in Source])
print(f"\nNASA POWER ({sum(catalog_df['source'] == 'NASA')} entries):")
print(catalog_df.loc[catalog_df["source"] == "NASA", "variable_id"].tolist())
print(f"\nCAMS ({sum(catalog_df['source'] == 'CAMS')} entries):")
print(catalog_df.loc[catalog_df["source"] == "CAMS", "variable_id"].tolist())
print(f"\nMERRA-2 ({sum(catalog_df['source'] == 'MERRA2')} entries):")
print(catalog_df.loc[catalog_df["source"] == "MERRA2", "variable_id"].tolist())
print(f"\nMODIS ({sum(catalog_df['source'] == 'MODIS')} entries):")
print(catalog_df.loc[catalog_df["source"] == "MODIS", "variable_id"].tolist())

In [ ]:
from susse.datasets import FeatureSelection
from susse.warehouse_ops.population.types import Source

# v1 recipe: aerosols + water vapour + cloudiness + NASA's pre-computed
# clearness_index (already a kt feature) on the NASA side, and CAMS
# `ghi_clear` on the CAMS side so NB 03 can derive `kt_cams = sat_ghi_cams
# / cams_ghi_clear`. Keeping NB 02 and NB 03 recipes in lockstep is the
# whole point of the recipe-vs-materialisation split: bumping either
# one means re-deriving the snapshot.
selection_v1 = FeatureSelection(
    nasa_variable_ids=(
        "aod_550", "precipitable_water", "cloud_amount", "clearness_index",
    ),
    cams_variable_ids=("ghi_clear",),
    merra_variable_ids=(),
    modis_variable_ids=(),
    include_satellite_irradiance=(Source.NASA_POWER, Source.CAMS),
    qc_levels=("pass",),
)
print("Recipe columns:", ("y_ghi_kwh_m2_day",) + selection_v1.aux_columns)
print("Recipe is empty?", selection_v1.is_empty)

## 2 — Build training pairs

`FeatureService.build_training_pairs` runs the SQL, applies the QC filter, and pivots each long-format aux table into source-prefixed wide columns. Note the **source prefix** on aux columns (`nasa_aod_550`, not `aod_550`) — when MERRA-2 lands its own `aod_550` analysis variable in v2, the names won't collide.

In [ ]:
from susse.datasets import FeatureService

fs = FeatureService(bq=bq)  # tables defaults to TableRefs(config=bq.config)
training_df = fs.build_training_pairs(
    selection=selection_v1,
    date_start=date(2024, 1, 1),
    date_end=date(2024, 12, 31),
)
print(f"Loaded {len(training_df):,} training rows x {len(training_df.columns)} columns.")
print("Columns:", list(training_df.columns))
training_df.head(3)

A quick sanity check: per-station coverage of the target column. Stations with zero rows here either have no QC-passed ground data in the requested window, or weren't ingested yet.

In [ ]:
training_df.groupby("location").size().sort_values(ascending=False).head(10)


## 3 — Build inference features

The same recipe drives inference. Given a `(lat, lon, date)`, `FeatureService.build_inference_features` returns a single-row frame with the satellite GHI estimates and the same auxiliary columns — but no target. This is what the portal will call in production.

In [ ]:
inference_today = fs.build_inference_features(
    selection=selection_v1,
    target_date=date(2024, 6, 15),
    lat=0.5179,
    lon=32.4715,  # central Uganda grid point
)
inference_today


## 4 — Dropping down to raw SQL when you need it

`FeatureService` covers the common cases (build training pairs, build
single-point inference features) but BigQuery is right there if you
need something it doesn't directly support — joining across sources in
unusual ways, computing aggregates over arbitrary windows, or just
sanity-checking what's in a table.

Three patterns worth knowing:

1. **Table references are typed**: use `tables.<table_name>` to get a
   fully-qualified table name, never hand-concatenate.
2. **`BigQueryClient.query()` returns a pandas DataFrame.** No
   streaming, no rows-iterator — for the warehouse scale (millions of
   rows), `pandas.DataFrame` is fine.
3. **For analytics you'll want to repeat, lift the SQL into
   `warehouse/sql/10_gold/` as a view** — both for re-use and so it
   shows up in the schema browser.

The cell below pulls Uganda-only monthly NASA + CAMS GHI averages
without going through `FeatureService`.

In [ ]:
# Example: monthly-mean satellite GHI per (geohash5, month) over
# Uganda only, 2022 — both NASA and CAMS, as named columns.
sql = f"""
SELECT geohash5,
       latitude, longitude,
       EXTRACT(MONTH FROM date) AS month,
       AVG(IF(source = 'NASA', ghi_kwh_m2_day, NULL)) AS nasa_ghi,
       AVG(IF(source = 'CAMS', ghi_kwh_m2_day, NULL)) AS cams_ghi,
       AVG(IF(source = 'CAMS', ghi_kwh_m2_day, NULL))
         - AVG(IF(source = 'NASA', ghi_kwh_m2_day, NULL))   AS cams_minus_nasa
FROM `{tables.irradiance_daily}`
WHERE date BETWEEN DATE('2022-01-01') AND DATE('2022-12-31')
  AND latitude  BETWEEN -1.5 AND 4.5    -- Uganda bbox
  AND longitude BETWEEN 29.5 AND 35.0
GROUP BY geohash5, latitude, longitude, month
"""
monthly = bq.query(sql)
print(f"{len(monthly):,} (geohash5, month) rows pulled.")
monthly.head()

## 5 — Materialise as a snapshot (local)

`build_and_write_snapshot` queries the warehouse, captures provenance (git SHA, susse version, per-source-table modification timestamps), writes `dataset.parquet` + `manifest.json` into the destination directory, and stamps the manifest with the parquet's SHA-256 hash.

In [ ]:
from susse.datasets import build_and_write_snapshot

SNAPSHOT_ROOT = (Path.cwd() / "../../data/training_snapshots").resolve()
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)

dataset_name = "susse_training_demo"
dataset_version = "v1-2024"
snapshot_dir = SNAPSHOT_ROOT / f"{dataset_name}_{dataset_version}"

snapshot = build_and_write_snapshot(
    feature_service=fs,
    selection=selection_v1,
    date_start=date(2024, 1, 1),
    date_end=date(2024, 12, 31),
    locations=None,
    name=dataset_name,
    version=dataset_version,
    dest=snapshot_dir,
)
print(f"Wrote {snapshot.manifest.n_rows} rows x {snapshot.manifest.n_cols} cols")
print(f"  to {snapshot_dir}")
print(f"  hash {snapshot.manifest.content_hash[:16]}...")

## 6 — Reload from local + verify

`load_snapshot` reads the parquet, reconstructs the manifest, and validates the parquet's SHA-256 against the manifest. Tampering or corruption raises a clear error before training silently consumes broken data.

In [ ]:
from susse.datasets import load_snapshot

reloaded = load_snapshot(snapshot_dir)
pd.testing.assert_frame_equal(reloaded.df, snapshot.df)
print(f"Roundtrip OK. {reloaded.manifest.name} {reloaded.manifest.version}")


## 7 — Inspect the manifest

The manifest is the single source of provenance truth for the snapshot. Every field is JSON-serialisable and survives the W&B upload as `manifest.json` *inside* the artifact (a subset is also mirrored into the artifact's `metadata` for the W&B UI).

In [ ]:
import json
print(json.dumps(reloaded.manifest.to_dict(), indent=2)[:2200])


## 8 — Log to W&B as an artifact

Set `LOG_TO_WANDB = True` to actually upload. Default is `False` so the notebook is safe to re-execute without hitting the W&B API on every cell run.

The upload creates a `dataset_build` run that records the recipe + warehouse state in its config, then attaches the snapshot directory as an artifact of type `training_dataset`. W&B assigns its own `v0`, `v1`, ... versioning on top of our human-readable `manifest.version`.

In [ ]:
LOG_TO_WANDB = False  # flip to True to upload
WANDB_PROJECT = "susse"
WANDB_ENTITY = None  # None = your default entity from `wandb login`

if LOG_TO_WANDB:
    from susse.datasets import log_dataset_artifact
    artifact_ref = log_dataset_artifact(
        snapshot_dir,
        artifact_name=dataset_name,
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        aliases=(dataset_version,),
    )
    print(f"Logged: {artifact_ref}")
else:
    artifact_ref = None
    print("LOG_TO_WANDB=False — skipping upload. Set True to push.")


## 9 — Reload from W&B + verify

`use_dataset_artifact` downloads the artifact contents and runs the same content-hash validation as the local loader. Inside a training run (NB 05), pass `run=` so the consumption shows up in the lineage graph.

In [ ]:
DOWNLOAD_FROM_WANDB = False  # flip to True after a successful upload above

if DOWNLOAD_FROM_WANDB and artifact_ref is not None:
    from susse.datasets import use_dataset_artifact
    download_dest = SNAPSHOT_ROOT / f"_wandb_download_{dataset_name}_{dataset_version}"
    redownloaded = use_dataset_artifact(artifact_ref, download_dest)
    pd.testing.assert_frame_equal(redownloaded.df, reloaded.df)
    print(f"W&B roundtrip OK. {redownloaded.manifest.name} {redownloaded.manifest.version}")
else:
    print(
        "DOWNLOAD_FROM_WANDB=False or no upload yet — skipping. "
        "Run cell 7 with LOG_TO_WANDB=True first."
    )


## What's next

Notebook 03 (`03_preprocessing.ipynb`) takes a `TrainingDataset` and applies preprocessing: clear-sky-index features (`kt = GHI / GHI_clear`), missing-value handling, train/val splits. The `Preprocessor` it produces will be stored alongside the model so inference uses identical transformations.

The **versioning discipline** to follow as the warehouse grows:

| Change | Action |
|---|---|
| Bug fix to one variable's curation | Apply migration → re-run recipe → bump materialisation version (`v1.1`) |
| New ground stations ingested | Re-run recipe → bump version (`v2`) |
| Add a new aux variable | New `FeatureSelection` → new artifact family |
| Smaller subset for an ablation | Derive a child recipe → separate artifact name |

The recipe itself never gets edited in place — that would silently change what `v1` means. New variants get new names.